# Train model 1 on high res data

Will produce a model that can then be used in next notebook to fill out all pixels.

#Run once

In [ ]:
# Cell 1 — repo + dependencies
!git clone https://github.com/fickas/crab_project.git /content/marsh-crab 2>/dev/null || \
 (cd /content/marsh-crab && git pull)
!pip install -r /content/marsh-crab/requirements.txt


#Wait until marsh-crab shows up locally

In [ ]:
# Cell 2 — Python setup
import sys
sys.path.insert(0, '/content/marsh-crab')
import marsh_utils as mu

##Run on changes

In [ ]:
# Run this cell whenever you edit the repo
!cd /content/marsh-crab && git pull
import importlib
importlib.reload(mu)

In [ ]:
# Standard scientific Python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import os
import json
import datetime

# Geospatial
import rasterio
from rasterio.mask import mask
from rasterio.features import rasterize
from rasterio.enums import Resampling
import geopandas as gpd
from shapely.geometry import box, mapping, Polygon
from shapely import wkt

# Image processing
import cv2
from PIL import Image

# Deep learning
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Segmentation library
# May need to install: !pip install segmentation-models-pytorch
import segmentation_models_pytorch as smp

# Augmentation
# May need to install: !pip install albumentations
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Metrics
from sklearn.metrics import confusion_matrix, classification_report

# Settings
torch.backends.cudnn.benchmark = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

In [ ]:
BATCH_SIZE = mu.recommended_batch_size()
BATCH_SIZE

In [ ]:
base = "/content/drive/MyDrive/salt_marsh/crab_project/WEL/"

In [ ]:
# Define paths to all expected input files
paths = {
    # Smoke test (existing data)
    "smoke_dem":         f"{base}/dem_folder/17May21_WEL_Low_P4_DEM.tif",
    "smoke_tile_labels": f"{base}/labels/master_grid.shp",

    # Production drone imagery (forthcoming)
    "ms_orthomosaic":    f"{base}/wellfleet_high_res/ms_5band.tif",       # 2 cm, B/G/R/RE/NIR stacked
    "pan_orthomosaic":   f"{base}/wellfleet_high_res/pan.tif",             # 1 cm, single band
    "pansharp_ms":       f"{base}/wellfleet_high_res/pansharp_5band.tif",  # 1 cm, B/G/R/RE/NIR stacked
    "dem_high_res":      f"{base}/wellfleet_high_res/dem_5m.tif",          # ?

    # Labels
    "crab_polygons":     f"{base}/wellfleet_high_res/crab_polygons.shp",

    # Optional: existing centerlines or other reference data
    "channel_centerlines": f"{base}/wellfleet_high_res/channel_centerlines.shp",

    # Output locations
    "runs_dir": f"{base}/wellfleet_high_res/runs",

}


#Synthetic data until have real


In [ ]:
synthetic = True

In [ ]:
if synthetic:
  paths["ms_orthomosaic"] =     f"{base}/synthetic_test/wellfleet_high_res_synthetic/ms_5band.tif"
  paths["pansharp_ms"] =        f"{base}/synthetic_test/wellfleet_high_res_synthetic/pansharp_5band.tif"  # 1 cm, B/G/R/RE/NIR stacked
  paths["dem_high_res"] =       f"{base}/synthetic_test/wellfleet_high_res_synthetic/dem_5m.tif"
  paths['pan_orthomosaic'] =    f'{base}/synthetic_test/wellfleet_high_res_synthetic/pan.tif'
  paths['crab_polygons'] =      f'{base}/synthetic_test/wellfleet_high_res_synthetic/crab_polygons.shp'
  paths['channel_centerlines'] = f'{base}/synthetic_test/wellfleet_high_res_synthetic/channel_centerlines.shp'
  paths['runs_dir']           = f'{base}/synthetic_test/wellfleet_high_res_synthetic/runs'

In [ ]:

# Create output directories
for key in ["runs_dir"]:
    os.makedirs(paths[key], exist_ok=True)

# Verify which input files exist (you'll have some, not others)
print("File existence check:")
for key, path in paths.items():
    exists = os.path.exists(path)
    status = "✓" if exists else "✗"
    print(f"  {status} {key}: {path}")

In [ ]:
EXPECTED = ['Blue', 'Green', 'Red', 'RedEdge', 'NIR']

with rasterio.open(paths["ms_orthomosaic"]) as src:
    print(src.count)         # 5
    print(src.descriptions)  # band names if the pipeline set them

    all_bands = src.read()           # whole raster, shape (5, H, W)
    nir_only  = src.read(5)          # just NIR, shape (H, W)
    print(all_bands.shape, nir_only.shape)

    if all(src.descriptions): assert list(src.descriptions) == EXPECTED, src.descriptions

##Compute derived bands and save to disk (or just load)

In [ ]:
paths = mu.ensure_indices(paths)
print(paths['ndvi'])
print(paths['ndre'])

##More bands
<pre>
Config.BAND_SPEC = [
    ('pan_orthomosaic', 1),
    ('ndvi',            1),
    ('slope',           1),      # ← swap in
    ('savi',            1),      # ← or add a 4th
]
</pre>

In [ ]:
# ── Spectral indices (read from pansharp_ms) ──
mu.ensure_savi(paths,       ms_key='pansharp_ms')
mu.ensure_evi(paths,        ms_key='pansharp_ms')
mu.ensure_gndvi(paths,      ms_key='pansharp_ms')
mu.ensure_ndwi(paths,       ms_key='pansharp_ms')
mu.ensure_ci_rededge(paths, ms_key='pansharp_ms')

# ── DEM-derived ──
mu.ensure_tpi(paths, dem_key='dem_high_res', neighborhood_m=0.05, out_key='tpi_micro')
mu.ensure_tpi(paths, dem_key='dem_high_res', neighborhood_m=0.3,  out_key='tpi_small')
mu.ensure_tpi(paths, dem_key='dem_high_res', neighborhood_m=2.0,  out_key='tpi_large')

# Optional, only if you want a non-redundant slope signal:
# mu.ensure_slope(paths, dem_key='dem_high_res', smooth_sigma_m=0.05, out_key='slope')

# QGIS visualization only — don't put in BAND_SPEC:
mu.ensure_hillshade(paths, dem_key='dem_high_res')

# ── Channel-dependent (order matters: ndwi → channel_mask → the rest) ──
mu.ensure_ndwi(paths, ms_key='pansharp_ms')               # if not already done
mu.ensure_channel_mask_from_ndwi(paths, ndwi_key='ndwi')  # needs ndwi
mu.ensure_distance_to_channel(paths, channel_mask_key='channel_mask')
mu.ensure_relative_elevation(paths, dem_key='dem_high_res',
                                    channel_mask_key='channel_mask')

# ── Texture (from pan or any single-band raster) ──
mu.ensure_local_std(paths,  src_key='pan_orthomosaic', window_m=0.3)
mu.ensure_laplacian(paths,  src_key='pan_orthomosaic')

# Texture from pan brightness
mu.ensure_local_range(paths, src_key='pan_orthomosaic', window_m=0.3, out_key='pan_range')

# Topographic micro-relief (burrows!) from DEM
mu.ensure_local_range(paths, src_key='dem_high_res',    window_m=0.3, out_key='dem_range')

# Entropy of pan = visual texture complexity
mu.ensure_local_entropy(paths, src_key='pan_orthomosaic', window_m=0.3, out_key='pan_entropy')

# Existing function — DEM roughness via std
mu.ensure_local_std(paths, src_key='dem_high_res', window_m=0.3, out_key='dem_roughness')

In [ ]:
new_keys = ['savi', 'evi', 'gndvi', 'ndwi', 'ci_rededge',
            'tpi_micro', 'tpi_small', 'tpi_large',
            'hillshade', 'channel_mask', 'dist_to_channel',
            'rel_elevation', 'local_std', 'laplacian']
missing = [k for k in new_keys if k not in paths]
print(f"Missing: {missing or 'none — all bands registered'}")

In [ ]:
class Config:
    # Class scheme — taken from marsh_utils (project-wide, not per-run)
    CLASS_COLUMN        = 'Class'
    CLASS_NAMES         = mu.CLASS_NAMES
    CLASSES             = mu.CLASSES
    CLASSES_OF_INTEREST = mu.CLASSES_OF_INTEREST
    PRIORITY            = mu.PRIORITY
    IGNORE_INDEX        = mu.IGNORE_INDEX
    QGIS_TO_MODEL       = mu.QGIS_TO_MODEL

    N_CLASSES = len(mu.CLASS_NAMES)

    SEED = mu.SEED

    # Per-run knobs (stay in notebook so you can edit and re-run)
    RESOLUTION_CM = 1.0
    BAND_SPEC    = [('pan_orthomosaic', 1), ('ndvi', 1), ('ndre', 1)]
    PATCH_SIZE   = 512
    OVERLAP      = 0.5
    BATCH_SIZE   = BATCH_SIZE
    EPOCHS        = 100
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY  = 1e-4
    NUM_WORKERS   = 4

    # Loss
    CE_WEIGHT   = 1.0
    DICE_WEIGHT = 1.0

    # Augmentation
    USE_D4_AUGMENTATION = True
    ENCODER      = 'efficientnet-b3'
    ENCODER_WEIGHTS = 'imagenet'

    # Splitting
    BLOCK_SIZE_M   = 3 if synthetic else 100
    TRAIN_FRAC     = 0.7
    VAL_FRAC       = 0.15
    TEST_FRAC      = 0.15
    REQUIRE_LABELS = True

    # Populated post-training (filled by pick_thresholds, etc.)
    CONFIDENCE_THRESHOLDS = None

In [ ]:
class Config2:
    """Configuration for Model 1 (high-res bank classifier)."""

    # Resolution and scale
    RESOLUTION_CM = 1.0
    PATCH_SIZE    = 512
    OVERLAP       = 0.5

    # Classes
    CLASS_COLUMN        = 'Class'
    CLASS_NAMES         = None          # fill in later
    CLASSES_OF_INTEREST = None          # fill in later
    N_CLASSES           = None          # fill in later
    PRIORITY            = None          # fill in later — polygon-overlap precedence
    IGNORE_INDEX        = 255

    # Band spec — (path-key, band-index) tuples, fed to build_patches_with_splits_multi
    BAND_SPEC = [
        ('pan_orthomosaic', 1),  #all 1 because all single band rasters - only band 1 exists
        ('ndvi',            1),
        ('ndre',            1),
    ]

    # Training
    BATCH_SIZE    = BATCH_SIZE
    EPOCHS        = 100
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY  = 1e-4
    NUM_WORKERS   = 4

    # Loss
    CE_WEIGHT   = 1.0
    DICE_WEIGHT = 1.0

    # Augmentation
    USE_D4_AUGMENTATION = True

    # Architecture
    ENCODER         = "efficientnet-b3"
    ENCODER_WEIGHTS = "imagenet"

    # Splitting
    BLOCK_SIZE_M   = 100
    TRAIN_FRAC     = 0.7
    VAL_FRAC       = 0.15
    TEST_FRAC      = 0.15
    REQUIRE_LABELS = True

    # Seeds
    SEED = 42

    # Inference — filled in after precision-coverage analysis
    CONFIDENCE_THRESHOLDS = None



In [ ]:
# Check what's in the orthomosaic

# Run on each available file
for key in ["ms_orthomosaic", "pan_orthomosaic", "dem_high_res"]:
    if os.path.exists(paths[key]):
        mu.inspect_raster(paths[key])

#Block-based spatial splits


The block ID is (x // block_size, y // block_size). Two patches in the same block always go to the same split. A patch crossing a block boundary gets assigned by its center, so up to ~patch_size / 2 worth of pixels can leak into an adjacent block's split — at 256px × 1cm GSD that's 1.28m of leakage near 100m block boundaries, which is negligible.

#Patch generator with split assignments

The two-pass structure is so the block assignment is computed over the full set of blocks the raster touches, before any patches are yielded. Otherwise an early yielded patch's split could depend on which blocks get seen later, which would not be deterministic.

In [ ]:
tiles_gdf = gpd.read_file(paths['crab_polygons'])

print(tiles_gdf.columns.tolist())
print(tiles_gdf.iloc[0])
print(tiles_gdf[Config.CLASS_COLUMN].value_counts())

In [ ]:
polys_gdf = tiles_gdf

###Map to 0-based

In [ ]:


# Map QGIS-side class values (1..6) to model class indices (0..5)
QGIS_TO_MODEL = {i:i-1 for i in range(1, len(Config.CLASS_NAMES)+1)}
print(f'{QGIS_TO_MODEL=}')

polys_gdf[Config.CLASS_COLUMN] = polys_gdf[Config.CLASS_COLUMN].map(QGIS_TO_MODEL)

# Catch unmapped values before they become silent bugs
if polys_gdf[Config.CLASS_COLUMN].isna().any():
    bad = polys_gdf.loc[polys_gdf[Config.CLASS_COLUMN].isna(), Config.CLASS_COLUMN].unique()
    raise ValueError(f"Unmapped class values from QGIS: {bad}")

polys_gdf[Config.CLASS_COLUMN] = polys_gdf[Config.CLASS_COLUMN].astype(int)

print(f"Class distribution:\n{polys_gdf[Config.CLASS_COLUMN].value_counts().sort_index()}")


###Filter classes

Decide if whether to collapse classes into a subset.

In [ ]:
filter_classes = False

In [ ]:
if filter_classes:
  original_class_column = polys_gdf[Config.CLASS_COLUMN].copy()
  new_labels_master = []
  mapping = dict(zip(range(0,5), [1,0,2,2,2]))  #3 class - 2 will be dropped
  for i,label in enumerate(original_class_column):
    mapped_label = mapping[label]
    if mapped_label==2: continue  #skip
    new_labels_master.append(mapped_label)

  CLASS_NAMES = {
    0: 'healthy_bank',
    1: 'eroding_non_crab',
    2: 'collapsed',
  }

  polys_gdf[Config.CLASS_COLUMN] = np.array(new_labels_master)
  print(f"Class distribution:\n{polys_gdf[Config.CLASS_COLUMN].value_counts().sort_index()}")

In [ ]:

# 1. Generate patches with splits
patches = list(mu.build_patches_with_splits_multi(
    paths=paths,
    polygons_gdf=polys_gdf,
    cfg=Config,
))

# 2. Sanity-check coverage
mu.summarize_patches(patches, class_names=Config.CLASS_NAMES)

# 3. Split into train/val/test
train_patches = [p for p in patches if p['split'] == 'train']
val_patches   = [p for p in patches if p['split'] == 'val']
test_patches  = [p for p in patches if p['split'] == 'test']

# 4. Compute (or load cached) normalization stats from training patches
stats_path = f"{base}/synthetic_test/wellfleet_high_res_synthetic/channel_stats.json" if synthetic else f"{base}/wellfleet_high_res/channel_stats.json"
channel_means, channel_stds = mu.compute_channel_stats(train_patches, stats_path)

# 5. Build datasets and loaders
train_ds = mu.MarshSegmentationDataset(train_patches, augmentation=mu.get_augmentations('train', channel_means, channel_stds))
val_ds   = mu.MarshSegmentationDataset(val_patches,   augmentation=mu.get_augmentations('val',   channel_means, channel_stds))
test_ds  = mu.MarshSegmentationDataset(test_patches,  augmentation=mu.get_augmentations('test',  channel_means, channel_stds))

train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True,  num_workers=Config.NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=Config.BATCH_SIZE, shuffle=False, num_workers=Config.NUM_WORKERS, pin_memory=True)

###What to look for in smoke test

Once you've run the summary, three things to look for in the output:

* Are split sizes roughly proportional to your fractions (70/15/15)? If one block is huge relative to others, you may get an imbalanced split.
* Is each class present in every split? Empty val classes mean noisy IoU for that class.
* What does labeled_fraction look like? If most patches are less than 10% labeled, the model is mostly seeing ignore — you may want to filter more aggressively or oversample patches with high labeled_fraction.

* Many fewer patches than expected → tile labels don't cover most of the DEM and require_labels=True is dropping them. Lower the threshold to accept partially-labeled patches, or set require_labels=False to see what's there.

* Empty classes in val or test → block split landed unluckily. Try a different seed, or check whether the offending class is concentrated in a small geographic area (in which case a different split strategy might be warranted).

##Get stats for later normalization

#Start of training

## Model + optimizer + scheduler

In [ ]:
import segmentation_models_pytorch as smp

LR          = Config.LEARNING_RATE
WEIGHT_DECAY = 1e-4

device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = smp.Unet(
    encoder_name=Config.ENCODER,
    encoder_weights=Config.ENCODER_WEIGHTS,
    in_channels=len(Config.BAND_SPEC),    # 3 for (Pan, NDVI, NDRE)
    classes=Config.N_CLASSES,
).to(device)

criterion = mu.CombinedLoss(
    num_classes=Config.N_CLASSES,
    ignore_index=255,
    ce_weight=1.0,
    dice_weight=1.0,
)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=Config.EPOCHS)

## Run it

In [ ]:
if synthetic:
  ckpt_path = f'{base}/synthetic_test/wellfleet_high_res_synthetic/best_model.pt'
else:
  ckpt_path = f'{base}/runs/best_model.pt'

In [ ]:
best_iou = mu.train(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    num_epochs=Config.EPOCHS,
    num_classes=Config.N_CLASSES,
    ignore_index=255,
    ckpt_path=ckpt_path,
    device=device,
    class_names=Config.CLASS_NAMES,
)
print(f"Best val mIoU: {best_iou:.4f}")

##Watch list

* Loss going down. If it doesn't decrease over the first 3-5 epochs, something's wrong with the loss/optimizer/data. Most likely culprits: NaN in normalized inputs, label values outside [0, num_classes-1] ∪ {ignore_index}, learning rate way off.

* Train mIoU gt val mIoU as expected. If val tracks train almost exactly, your block-based splits aren't separating well enough — leakage. If val crashes far below train, you're overfitting and want more augmentation or fewer epochs.

* Per-class IoU. Rare classes (collapsed, eroding_non_crab maybe) will have noisier IoU. If a class stays at 0.0 IoU across epochs, the model isn't learning it — usually because there are too few pixels of it in train, or your class weights are pushing the model away from it.

##Need more than mIoU

The most useful metric isn't mIoU on the test set — it's precision at high-confidence thresholds. Specifically: "for pixels where Model 1 predicts class 3 with softmax ≥ 0.7, what fraction actually are class 3?" That high-confidence subset is what becomes Model 2's training data, and any false positives there poison Model 2.

So in addition to training-time mIoU, the right evaluation summary for Model 1 is a precision-coverage curve:

* x-axis: confidence threshold (0.5, 0.6, ..., 0.95)
* y-axis: per-class precision at that threshold

Annotated with coverage (what fraction of the marsh you keep at each threshold)

This tells you exactly how strict the threshold needs to be to get acceptably clean labels for Model 2, and how much area you sacrifice at each strictness level. If at threshold 0.7 you get 95% precision on class 3 but only cover 12% of the marsh, that might still be enough labels for Model 2 because the marsh is big.

In [ ]:
# ============ After train() completes ============

# 1. Reload the best checkpoint into the model (in case last epoch wasn't best)
best_ckpt = torch.load(ckpt_path, map_location=device)
state_dict = best_ckpt.get('model_state_dict', best_ckpt) if isinstance(best_ckpt, dict) else best_ckpt
model.load_state_dict(state_dict)
print(f"Loaded checkpoint from epoch {best_ckpt['epoch']}, "
      f"val mIoU = {best_ckpt['best_iou']:.4f}")
print(f"Per-class IoU: {best_ckpt['iou_per_class']}")

# 2. Run precision-coverage on the validation set to choose thresholds
print("\nPrecision-coverage analysis on val set:")
pc_results_val = mu.evaluate_precision_coverage(
    model=model,
    loader=val_loader,
    num_classes=Config.N_CLASSES,
    ignore_index=Config.IGNORE_INDEX,
    device=device,
)

# Visualize so you can sanity-check before committing thresholds
mu.plot_precision_coverage(
    pc_results_val,
    class_names=Config.CLASS_NAMES,
    classes_of_interest=Config.CLASSES_OF_INTEREST,
)
mu.plot_precision_vs_coverage(
    pc_results_val,
    class_names=Config.CLASS_NAMES,
    classes_of_interest=Config.CLASSES_OF_INTEREST,
)
mu.print_precision_coverage_table(
    pc_results_val,
    class_names=Config.CLASS_NAMES,
    classes_of_interest=Config.CLASSES_OF_INTEREST,
)

# 3. Auto-pick thresholds at a precision floor of your choice
TARGET_PRECISION = 0.9
print(f"\nChosen thresholds (target precision = {TARGET_PRECISION}):")
Config.CONFIDENCE_THRESHOLDS = mu.pick_thresholds(
    pc_results_val,
    classes_of_interest=Config.CLASSES_OF_INTEREST,
    target_precision=TARGET_PRECISION,
)

# 4. (Optional) Verify on the held-out test set so the precision numbers
#    you report aren't from the same data you tuned thresholds on
print("\nVerification on test set (using thresholds chosen from val):")
pc_results_test = mu.evaluate_precision_coverage(
    model=model,
    loader=test_loader,
    num_classes=Config.N_CLASSES,
    ignore_index=Config.IGNORE_INDEX,
    device=device,
)
mu.print_precision_coverage_table(
    pc_results_test,
    class_names=Config.CLASS_NAMES,
    classes_of_interest=Config.CLASSES_OF_INTEREST,
)


In [ ]:

# 5. NOW save artifacts — Config.CONFIDENCE_THRESHOLDS is populated
from datetime import datetime
if synthetic:
  artifact_dir = f'{base}/synthetic_test/wellfleet_high_res_synthetic/runs/run_{datetime.now():%Y-%m-%d_%H-%M-%S}'
else:
  artifact_dir = f'{base}/runs/run_{datetime.now():%Y-%m-%d_%H-%M-%S}/'

os.makedirs(artifact_dir, exist_ok=True)
print(f"Run output: {artifact_dir}")

mu.save_training_artifacts(
    output_dir=artifact_dir,
    model=model,
    channel_means=channel_means,
    channel_stds=channel_stds,
    training_summary={
        'best_val_miou':     float(best_iou),
        'iou_per_class':     best_ckpt['iou_per_class'],
        'num_train_patches': len(train_patches),
        'num_val_patches':   len(val_patches),
        'num_test_patches':  len(test_patches),
        'target_precision':  TARGET_PRECISION,
    },
    cfg=Config,
)

###How to interpret the output for your Model 2 labeling decision

The tradeoff plot is the actionable one. Each line is a class; each point is a confidence threshold. The point you want to pick for that class's Model 2 labels is the one where precision is "high enough" (say 0.9+) at the highest possible coverage. If class 3 hits 0.95 precision at coverage 0.05 (5% of marsh pixels labeled), and class 5 only hits 0.7 precision at any threshold, you'd commit class-3 labels to Model 2 confidently and either gather more data for class 5 or accept noisier class-5 labels.

The threshold annotations on the tradeoff plot make this concrete: "to get class 3 at precision ≥ 0.9, threshold at 0.75, which keeps 4% of marsh pixels." That's a sentence you can put in a report.

You might also find that different classes need different thresholds — that's expected, and the per-class precision curve is exactly the right tool for setting them. The standard "single threshold for all classes" rule of thumb (e.g., 0.7) is a starting point, not the final answer.

#Analyze channel importance

In [ ]:
CHANNEL_NAMES = ['0', '1', '2']

In [ ]:

# Quick pass: zero ablation
zero_results = mu.channel_zero_ablation(
    model=model, loader=val_loader,
    num_classes=Config.N_CLASSES,
    classes_of_interest=Config.CLASSES_OF_INTEREST,
    channel_names=CHANNEL_NAMES,
    device=device,
)

print(f"Baseline mIoU: {zero_results['baseline_mean_iou']:.4f}")
for name, r in zero_results['per_channel'].items():
    print(f"  zero {name:6s}: mIoU={r['mean_iou']:.4f}  drop={r['drop_from_base']:+.4f}")

# Slower pass: permutation importance
perm_results = mu.channel_permutation_importance(
    model=model, loader=val_loader,
    num_classes=Config.N_CLASSES,
    classes_of_interest=Config.CLASSES_OF_INTEREST,
    channel_names=CHANNEL_NAMES,
    device=device,
    n_repeats=3,
)

print(f"\nPermutation importance (mean ± std over {3} repeats):")
for name, r in perm_results['per_channel'].items():
    print(f"  permute {name:6s}: drop={r['mean_drop']:+.4f} ± {r['std_drop']:.4f}")

##How to interpret the output:

* Big drop when zeroed → model relies heavily on that channel (could be true importance or learned reliance)
* Small drop when zeroed → model isn't using that channel much (might be genuinely useless, or might be that the model leans on other correlated channels)
* Big drop when permuted, small drop when zeroed → model is using the channel's spatial structure but is partially robust to scaling
* Both large → channel is genuinely important
* Both small → channel is probably redundant

A useful pattern is comparing the two: if zeroing gives a big drop but permutation gives a small one, the model is leaning on the channel's value range more than its spatial pattern. This sometimes flags problems — e.g., the model has learned "if NDVI value is low, prediction is X" without using the spatial pattern, which is a clue that the channel is acting more like a scalar gate than a spatial feature.

###When to escalate to retrain ablation:

If zero+permutation suggest a channel might be droppable (mean IoU drop < ~0.01 across both tests), do one retrain without that channel and compare the new baseline. If the retrained model matches the original, drop the channel. If the retrained model loses meaningful IoU even though the inference ablation suggested the channel was unused, the trained model was using it in ways the inference tests didn't capture. Retrain is more expensive but it's the only test that tells you whether a channel should have been there during training.

For the Pan/NDVI/NDRE setup specifically, my prior is that NDVI does the heaviest lifting (vegetation contrast is the cleanest signal), NDRE adds modest value (matters for stress within the vegetated regime), and pan adds spatial resolution that the others can't replace. But that prior is worth nothing compared to actually running the ablation once you have data.

In [ ]:
results = mu.channel_permutation_importance_per_class(
    model, val_loader, num_classes=6,
    ignore_index=Config.IGNORE_INDEX,
    n_repeats=3,
    device=device,
    class_names=Config.CLASS_NAMES,                # dict {0:'other', 1:'healthy_bank', ...}
    band_names=[b[0] for b in Config.BAND_SPEC],   # ['pan_orthomosaic', 'ndvi', 'ndre']
)
# results['drops_mean'][ch, c]   → drop for channel ch on class c
# results['drops'][ch, c, :]     → per-repeat drops (for variance / significance testing)

#Experiment with different thresholding approaches

In [ ]:
def predict_probs(model, loader, ignore_index=255, device='cuda'):
    """Run the model and return per-pixel (probs, true_label).

    Returns:
      probs: (N, num_classes) float32 softmax probabilities
      true:  (N,) int64 ground truth labels
    """
    import torch, numpy as np
    model.eval()
    all_probs, all_true = [], []
    with torch.no_grad():
        for batch in loader:
            # Support both tuple and dict batch formats
            if isinstance(batch, dict):
                x, y = batch['image'], batch['mask']
            else:
                x, y = batch[0], batch[1]
            x = x.to(device)
            y = y.to(device)
            logits = model(x)
            probs  = torch.softmax(logits, dim=1).permute(0, 2, 3, 1)   # (B,H,W,C)
            mask   = (y != ignore_index)
            all_probs.append(probs[mask].cpu().numpy())
            all_true.append(y[mask].cpu().numpy())
    return np.concatenate(all_probs), np.concatenate(all_true)

def rule_argmax(probs):
    return probs.argmax(axis=1)

def rule_soft_cascade(probs, bank_threshold=0.5, bank_classes=(1, 2, 3, 4, 5)):
    """If sum of bank probs > threshold, argmax among bank classes; else 'other'."""
    import numpy as np
    p_bank_sum = probs[:, list(bank_classes)].sum(axis=1)
    bank_mask  = np.zeros(probs.shape[1], dtype=bool); bank_mask[list(bank_classes)] = True
    masked = np.where(bank_mask[None, :], probs, -np.inf)
    bank_argmax = masked.argmax(axis=1)
    return np.where(p_bank_sum > bank_threshold, bank_argmax, 0)

def rule_priority(probs, thresholds, priority, default_threshold=0.5):
    """Walk classes in priority order; first to pass its threshold wins.
    Classes without an explicit threshold use `default_threshold`.
    Accepts thresholds as a dict OR an array."""
    import numpy as np
    pred = np.zeros(probs.shape[0], dtype=np.int64)
    decided = np.zeros(probs.shape[0], dtype=bool)
    for cls in priority:
        if isinstance(thresholds, dict):
            thr = thresholds.get(cls, default_threshold)
        else:
            thr = thresholds[cls] if cls < len(thresholds) else default_threshold
        passes = (probs[:, cls] > thr) & ~decided
        pred[passes] = cls
        decided     |= passes
    return pred

In [ ]:
import sklearn

# Compute probs once
probs, true = predict_probs(model, val_loader, ignore_index=Config.IGNORE_INDEX, device=device)

# Try different rules — fast since model doesn't run again
for rule_name, rule_fn in [
    ('argmax',             lambda p: rule_argmax(p)),
    ('soft_cascade_05',    lambda p: rule_soft_cascade(p, bank_threshold=0.5)),
    ('soft_cascade_07',    lambda p: rule_soft_cascade(p, bank_threshold=0.7)),
    ('priority_all',       lambda p: rule_priority(p, Config.CONFIDENCE_THRESHOLDS, Config.PRIORITY)),
    ('priority_crab_only', lambda p: rule_priority(p, Config.CONFIDENCE_THRESHOLDS, [5, 4, 3])),
]:
    pred = rule_fn(probs)
    cm = sklearn.metrics.confusion_matrix(true, pred, labels=range(len(Config.CLASS_NAMES)))
    print(f"\n=== {rule_name} ===")
    mu.display_confusion_matrix(cm, list(Config.CLASS_NAMES.values()), normalize='recall')

#Look at confidence levels

In [ ]:
import sklearn.metrics
import numpy as np
import collections


# ─── Helper: evaluate one rule (handles abstention) ───
def evaluate_rule(true, pred, class_names, abstain_label=255):
    """Show confusion matrix for non-abstained pixels, report abstention rate."""
    valid = pred != abstain_label
    n_abstained = int((~valid).sum())
    pct = 100 * n_abstained / len(pred) if len(pred) else 0.0
    print(f"  Pixels: {valid.sum():,}/{len(pred):,} kept  ·  "
          f"{n_abstained:,} abstained ({pct:.1f}%)")
    if valid.sum() == 0:
        print("  (no predictions to score)")
        return
    cm = sklearn.metrics.confusion_matrix(
        true[valid], pred[valid], labels=range(len(class_names))
    )
    mu.display_confusion_matrix(cm, class_names, normalize='recall')


# ─── All rules to try ───
class_names = list(Config.CLASS_NAMES.values())

rules = [
    # Pure argmax (no thresholding, no abstain)
    ('argmax',
        lambda p: rule_argmax(p)),

    # Argmax + abstain on low max-confidence
    ('argmax_abstain_05',
        lambda p: mu.rule_argmax_abstain(p, min_confidence=0.5)),
    ('argmax_abstain_07',
        lambda p: mu.rule_argmax_abstain(p, min_confidence=0.7)),

    # Argmax + abstain on close top-2 (catches "tied" pixels)
    ('margin_abstain_015',
        lambda p: mu.rule_margin_abstain(p, min_margin=0.15)),
    ('margin_abstain_025',
        lambda p: mu.rule_margin_abstain(p, min_margin=0.25)),

    # Argmax + abstain on high entropy (catches spread distributions)
    ('entropy_abstain_15',
        lambda p: mu.rule_entropy_abstain(p, max_entropy=1.5)),

    # Soft cascade — sum P(banks) > threshold, then argmax among banks
    ('soft_cascade_05',
        lambda p: rule_soft_cascade(p, bank_threshold=0.5)),
    ('soft_cascade_07',
        lambda p: rule_soft_cascade(p, bank_threshold=0.7)),

    # Priority — walk classes in PRIORITY order; first to pass threshold wins
    ('priority_all',
        lambda p: rule_priority(p, Config.CONFIDENCE_THRESHOLDS, Config.PRIORITY)),
    ('priority_crab_only',
        lambda p: rule_priority(p, Config.CONFIDENCE_THRESHOLDS, [5, 4, 3])),
]


# ─── Run each rule ───
for name, rule_fn in rules:
    print(f"\n{'='*70}\n=== {name} ===\n{'='*70}")
    pred = rule_fn(probs)
    evaluate_rule(true, pred, class_names)

In [ ]:
print(f"\n{'='*70}\n=== margin tie diagnostics ===\n{'='*70}")
sorted_idx = np.argsort(probs, axis=1)
sorted_probs = np.take_along_axis(probs, sorted_idx, axis=1)
margin = sorted_probs[:, -1] - sorted_probs[:, -2]

for thresh in [0.15, 0.25]:
    tied = margin < thresh
    print(f"\nmargin < {thresh}: {tied.sum():,} pixels ({100*tied.mean():.1f}%)")
    if tied.sum() == 0:
        continue
    top1 = sorted_idx[:, -1][tied]
    top2 = sorted_idx[:, -2][tied]
    pairs = [tuple(sorted([t1, t2])) for t1, t2 in zip(top1, top2)]
    counter = collections.Counter(pairs)
    print("  Top tied pairs (class_a ↔ class_b → count):")
    for (a, b), n in counter.most_common(5):
        print(f"    {class_names[a]:18s} ↔ {class_names[b]:18s}  {n:,}")

<img src='https://www.dropbox.com/scl/fi/oyu1fdcvn21iejfclvunu/Screenshot-2026-06-08-at-11.58.13-AM.png?rlkey=kkf2krz8plyov1mcfxapqswty&dl=1'>

In [ ]:
foobar()

#Save for use in production